# 8 · Materials & boundaries — in 2D and 3D

Real problems live on domains made of **parts**: different materials, and boundaries with
different conditions. NGSolve attaches everything to **named regions** — and the workflow
is **identical in 2D and 3D**. We meet it first on a flat **🍪 cookie** (2D), then on a
**☕ cup of coffee** (3D), and finish by solving a PDE on **just one subdomain**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

## 1. A 2-D geometry with names

We give the chocolate chips the **material** name `"chip"` and the rest `"dough"`, and we
name the outer **boundary** `"rim"`. `Glue` keeps the chips as separate subdomains inside
the cookie. The mesh can then be **queried** for its materials and boundaries, and any
region selected with `mesh.Materials(...)` / `mesh.Boundaries(...)` (regular expressions
allowed) and measured with `Integrate`.

In [ ]:
def cookie_with_chips():
    cookie = WorkPlane().Circle(0, 0, 3.0).Face()
    cookie.edges.name = "rim"                            # name the outer boundary
    chips = []
    for (cx, cy, r) in [(-1.2, 0.8, 0.5), (1.0, 1.1, 0.45), (0.2, -1.3, 0.55),
                        (1.5, -0.8, 0.4), (-1.3, -0.9, 0.45)]:
        chip = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, r).Face()
        chip.name = "chip"                               # material name
        chips.append(chip)
    dough = cookie
    for chip in chips:
        dough = dough - chip
    dough.name = "dough"
    return Glue([dough] + chips)

mesh = Mesh(OCCGeometry(cookie_with_chips(), dim=2).GenerateMesh(maxh=0.4))
print("materials :", set(mesh.GetMaterials()))
print("boundaries:", set(mesh.GetBoundaries()))
print("dough area:", round(Integrate(CF(1), mesh, definedon=mesh.Materials("dough")), 2),
      " chip area:", round(Integrate(CF(1), mesh, definedon=mesh.Materials("chip")), 2))

## 2. Piecewise coefficients, used in a PDE

`mesh.MaterialCF` builds a `CoefficientFunction` taking a **different value per material**
— here the heat conductivity $\kappa$ (chocolate conducts far better than dough). It drops
straight into the weak form; `dirichlet="rim"` uses the boundary name. The temperature
comes out **flatter over the well-conducting chips**.

In [ ]:
kappa = mesh.MaterialCF({"chip": 50.0, "dough": 1.0})
Draw(kappa, mesh, "κ — piecewise conductivity")

fes = H1(mesh, order=2, dirichlet="rim")
u, v = fes.TnT()
a = BilinearForm(kappa * grad(u) * grad(v) * dx).Assemble()
f = LinearForm(1 * v * dx).Assemble()
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
Draw(gfu, mesh, "temperature (flatter over the chips)")

## 3. Solving on **one subdomain** only

Sometimes the PDE only lives on *part* of the domain. Passing `definedon=mesh.Materials(...)`
to the **space** builds degrees of freedom *only* there, and restricting the integrals with
`dx(definedon=…)` assembles only over that region. Here we heat **just the dough** (the chips
are cut out, their interface left as a free/Neumann boundary), and hold the `rim` fixed.

In [ ]:
dough = mesh.Materials("dough")
fesd = H1(mesh, order=2, definedon=dough, dirichlet="rim")
print(f"sub-space on the dough: {fesd.ndof} dofs  (vs {fes.ndof} on the whole cookie)")
ud, vd = fesd.TnT()
ad = BilinearForm(grad(ud) * grad(vd) * dx(definedon=dough)).Assemble()
fd = LinearForm(1 * vd * dx(definedon=dough)).Assemble()
gfd = GridFunction(fesd)
gfd.vec.data = ad.mat.Inverse(fesd.FreeDofs(), inverse="sparsecholesky") * fd.vec
Draw(gfd, mesh, "solved on the dough only (chips are holes)")

## 4. A subtle bug — the misspelled region

A **wrong** region name is *not* an error in NGSolve — it simply selects **nothing**, and
the integral is silently **zero**. Your code runs; the result is wrong. Always sanity-check
names against `mesh.GetMaterials()` / `mesh.GetBoundaries()`.

In [ ]:
print("area of Materials('chips') [typo for 'chip']:",
      Integrate(CF(1), mesh, definedon=mesh.Materials("chips")))      # → 0.0 !

## 5. The same ideas in 3-D — a cup of coffee ☕

Nothing changes going to 3D. We glue a **ceramic** mug (with a ring handle) around a body of
**coffee**, and name the faces (`bottom`, `wall`, `rim`, `handle`, and the coffee `surface`).
A `MaterialCF` gives the **volumetric** conductivity $\kappa$; a `BoundaryCF` gives a
**surface** coefficient $\alpha$ (the heat-transfer rate). The mug stands on a **hot coaster**
(Dirichlet, $80^\circ$) and loses heat to the air everywhere else by **Newton cooling**
$\alpha(T-T_{\text{air}})$ — a genuinely **mixed** boundary condition, all from named regions.

In [ ]:
def coffee_cup_3d():
    R, ri, Hout, base, fill = 4.0, 3.4, 9.0, 1.0, 6.0
    outer  = Cylinder(Pnt(0, 0, 0), Z, r=R,  h=Hout, bottom="bottom", mantle="wall")
    cavity = Cylinder(Pnt(0, 0, base), Z, r=ri, h=Hout)
    ceramic = outer - cavity
    ceramic.faces.Max(Z).name = "rim"
    prof = WorkPlane(Axes((0, 0, 0), n=Y)).Circle(1.6, 0, 0.45).Face()
    handle = prof.Revolve(Axis(Pnt(0, 0, 0), Z), 360) \
                 .Rotate(Axis(Pnt(0, 0, 0), X), 90).Move((R + 0.9, 0, Hout/2))
    handle = (handle - cavity); handle.faces.name = "handle"
    ceramic = ceramic + handle; ceramic.solids.name = "ceramic"
    ceramic.faces.col = (0.85, 0.83, 0.78)
    coffee = Cylinder(Pnt(0, 0, base), Z, r=ri, h=fill - base, top="surface")
    coffee.solids.name = "coffee"; coffee.faces.col = (0.40, 0.26, 0.13)
    return Glue([ceramic, coffee])

clip3d = {"Clipping": {"enable": True, "function": True, "x": 0, "y": 1, "z": 0, "dist": 0}}
mesh3 = Mesh(OCCGeometry(coffee_cup_3d()).GenerateMesh(maxh=1.0)); mesh3.Curve(2)
print("3D materials :", mesh3.GetMaterials())
print("3D boundaries:", set(mesh3.GetBoundaries()))

kappa3 = mesh3.MaterialCF({"ceramic": 1.5, "coffee": 0.6})
alpha  = mesh3.BoundaryCF({"wall": 6.0, "surface": 10.0}, default=1.5)   # small, not zero
Tair, Thot = 20.0, 80.0

fes3 = H1(mesh3, order=2, dirichlet="bottom")
u, v = fes3.TnT()
a3 = BilinearForm(kappa3 * grad(u) * grad(v) * dx + alpha * u * v * ds).Assemble()
f3 = LinearForm(alpha * Tair * v * ds).Assemble()
gf3 = GridFunction(fes3)
gf3.Set(Thot, definedon=mesh3.Boundaries("bottom"))         # hot coaster
gf3.vec.data += a3.mat.Inverse(fes3.FreeDofs(), inverse="sparsecholesky") * (f3.vec - a3.mat * gf3.vec)
Draw(gf3, mesh3, "temperature", settings=clip3d)

## 6. Reading off region quantities

`definedon=mesh.Materials(...)` restricts any integral to one material — the **average
temperature of the coffee** versus the ceramic, or the coffee's volume, without touching
the rest of the mesh. (The better-conducting ceramic runs warmer near the coaster; swap the
two `kappa` values and the picture flips.)

In [ ]:
def average(region):
    reg = mesh3.Materials(region)
    return Integrate(gf3, mesh3, definedon=reg) / Integrate(CF(1), mesh3, definedon=reg)

print(f"coffee volume       : {Integrate(CF(1), mesh3, definedon=mesh3.Materials('coffee')):.1f}")
print(f"average T in coffee : {average('coffee'):.1f} °C")
print(f"average T in ceramic: {average('ceramic'):.1f} °C")

Next: with regions and boundaries in hand, a full steady **heat** problem in the cup.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("07-saddle-point", "7 · Saddle-point problems 🐎")
    _next = ("09-steady-heat", "9 · Steady heat in the cup ☕")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))